# Modèle v5 : optimisation et comparaison
## Reconnaissance du genre - noms propres ivoiriens | SIGAP / AFOR
**INP-HB · ESI · Stage 2A · 2025-2026**

Objectif : améliorer le CNN + BiLSTM + MHA (v4) pour qu'il dépasse une régression
logistique bien réglée (val acc 0,9285 ; test acc 0,9393 contre 0,9331 pour le v4).

**Protocole**
- Même découpage train / val / test que `train_model.ipynb`, **figé** dans `split_reference.csv`.
- Tous les choix (augmentation, architecture, hyperparamètres) se font sur la **validation**.
- Le **test** n'est évalué qu'une seule fois (section 11), avec 3 graines et un test de McNemar.

**Architectures**
- **v4** : CNN + BiLSTM + MHA sur la chaîne complète (modèle du mémoire).
- **v5** : modèle « par mot ». Chaque mot est encodé par ses caractères (Conv1D k=2,3,4)
  et par son identité (embedding de mot, remplacé au hasard par `<UNK>` à l'entraînement).
  Les mots échangent ensuite de l'information par attention **sans encodage de position** :
  le modèle est invariant à l'ordre NOM / PRÉNOMS par construction.

**Augmentation**
- `v4` : celle du notebook d'origine (permutation, inversion, retrait d'un mot 30 %, bruit 15 %).
- `none` : aucune.
- `safe` : permutation + retrait d'un mot, **sans jamais retirer le dernier prénom**
  (un nom de famille seul ne porte pas l'information de genre), pas de bruit caractère.
- `noise` : `safe` + inversion de deux caractères.

> Environnement : noyau Python de WSL (GPU), créé avec `uv sync --group train`.
> Avant de commiter, vider les sorties du notebook.

## 0. Paramètres

Cellule marquée `parameters` : papermill peut la remplacer pour lancer les calculs longs
sans ouvrir le notebook, par exemple :

```bash
papermill optimisation.ipynb runs/optimisation_nuit.ipynb -p RUN_ABLATION True -p RUN_OPTUNA True
```

In [ ]:
# Expériences à lancer (les plus longues sont désactivées par défaut)
RUN_LR       = True
RUN_SINGLE   = True
RUN_ABLATION = False   # 8 entraînements
RUN_OPTUNA   = False   # N_TRIALS entraînements (avec élagage)
RUN_FINAL    = False   # à lancer UNE fois, configuration figée
N_TRIALS     = 30
OPTUNA_TIMEOUT_H = None  # ex. 7.5 : plus de nouvel essai Optuna après 7 h 30
FINAL_SEEDS  = [42, 43, 44]
FINAL_USE_BEST = True  # True : runs/best_v5.json ; False : configuration CFG (ex. v4)
FINAL_DIR    = 'runs/final'  # probabilités val + test par graine, pour evaluation_finale.ipynb
CFG_OVERRIDES = {}     # ex. {'epochs': 1} pour un test rapide
RUNS_DIR     = 'runs'
MLFLOW_URI   = 'sqlite:///mlflow.db'
SMOKE        = False   # test technique : le jeu de test est remplacé par la validation

## 1. Configuration et suivi MLflow

Chaque entraînement est enregistré dans MLflow (paramètres, courbes par époque, métriques).
Aucun nom n'est enregistré : ni le dataset, ni les prédictions.

Pour consulter les résultats (dans WSL, puis http://localhost:5000 dans le navigateur) :

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [ ]:
import os, re, json, time, unicodedata, logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from collections import Counter

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers as L
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import mlflow

print(f"TensorFlow {tf.__version__} | GPU : {tf.config.list_physical_devices('GPU')}")

DATASET_PATH = 'dataset_nettoye.csv'
MAX_TOKENS   = 8       # 8 mots couvrent 25 910 / 25 911 noms
MAX_CHARS    = 14      # P99.9 de la longueur d'un mot
os.makedirs(RUNS_DIR, exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_URI)
logging.getLogger('mlflow').setLevel(logging.WARNING)
logging.getLogger('alembic').setLevel(logging.WARNING)

CFG = dict(
    arch='v5', aug='safe', n_aug=5, word_drop=0.30, prenoms_first=0.50,
    char_noise=0.15,
    # v4
    embed_dim=96, cnn_filters=128, lstm1=128, lstm2=64, mha_heads=4, mha_key_dim=32,
    # v5
    char_dim=48, word_dim=64, tok_dim=128, n_attn_layers=1, word_min_freq=2, unk_prob=0.20,
    use_words=True, word_dropout=0.0,
    # commun
    dropout=0.3, lr=1e-3, batch_size=128, epochs=20, label_smooth=0.0, patience=5,
    weight_decay=1e-4,
)
CFG.update(CFG_OVERRIDES)

## 2. Données et découpage

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    text = re.sub(r"['-]", ' ', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(DATASET_PATH)
df['nom_clean']     = df['NOM'].apply(clean_text)
df['prenoms_clean'] = df['PRENOMS'].fillna('').apply(clean_text)
df['full_name']     = (df['nom_clean'] + ' ' + df['prenoms_clean']).str.strip()
df = df[df['full_name'] != ''].reset_index(drop=True)
df['label'] = (df['GENRE'] == 'M').astype(int)

# Découpage FIGÉ : calculé une fois sur les étiquettes d'origine (voir split_reference.csv).
# Le recalculer après une correction d'étiquettes déplacerait des noms entre les ensembles,
# car le découpage est stratifié sur le genre.
SPLIT_PATH = 'split_reference.csv'

def load_split(df):
    import hashlib
    if not os.path.exists(SPLIT_PATH):
        raise FileNotFoundError(f"{SPLIT_PATH} introuvable : il fige le découpage train/val/test")
    sp = pd.read_csv(SPLIT_PATH)
    emp = df['full_name'].map(lambda s: hashlib.sha1(s.encode()).hexdigest()[:10]).values
    assert len(sp) == len(df) and (sp['empreinte'].values == emp).all(), \
        f"{SPLIT_PATH} ne correspond pas au dataset (ordre ou contenu des noms modifié)"
    return [df.index[sp['ensemble'].values == e] for e in ('train', 'val', 'test')]

idx_tr, idx_va, idx_te = load_split(df)
df_tr, df_va, df_te = df.loc[idx_tr], df.loc[idx_va], df.loc[idx_te]
if SMOKE:
    df_te = df_va
    print("SMOKE : le jeu de test est remplacé par la validation (test technique)")

# Aucun nom n'est affiché : seulement des effectifs
print(f"Train={len(df_tr):,} | Val={len(df_va):,} | Test={len(df_te):,}")

## 3. Augmentation des données

Correction principale par rapport au v4 : le retrait d'un mot ne supprime jamais le dernier
prénom. Dans le v4, « kouassi marie » pouvait devenir « kouassi » en restant étiqueté F,
ce qui apprend au modèle qu'un nom de famille indique le genre.

In [ ]:
def char_noise(word, rng):
    if len(word) < 3: return word
    i = int(rng.integers(0, len(word)-1))
    chars = list(word)
    chars[i], chars[i+1] = chars[i+1], chars[i]
    return ''.join(chars)


def augment_row(nom_parts, prenoms_parts, cfg, rng):
    """Une variante augmentée d'un nom, selon cfg['aug']."""
    if cfg['aug'] == 'v4':
        parts = nom_parts + prenoms_parts
        rng.shuffle(parts)
        if prenoms_parts and rng.random() < cfg['prenoms_first']:
            p_s = prenoms_parts.copy(); rng.shuffle(p_s)
            n_s = nom_parts.copy();     rng.shuffle(n_s)
            parts = p_s + n_s
        if len(parts) > 1 and rng.random() < cfg['word_drop']:
            idx = int(rng.integers(0, len(parts)))
            parts = [p for i, p in enumerate(parts) if i != idx]
        if parts and rng.random() < cfg['char_noise']:
            idx = int(rng.integers(0, len(parts)))
            parts[idx] = char_noise(parts[idx], rng)
        return parts

    # safe / noise : on ne retire jamais le dernier prénom restant
    noms, pres = list(nom_parts), list(prenoms_parts)
    if rng.random() < cfg['word_drop']:
        candidates = [('n', i) for i in range(len(noms)) if len(noms) + len(pres) > 1]
        candidates += [('p', i) for i in range(len(pres)) if len(pres) > 1]
        if candidates:
            kind, i = candidates[int(rng.integers(0, len(candidates)))]
            (noms if kind == 'n' else pres).pop(i)
    parts = noms + pres
    rng.shuffle(parts)
    if cfg['aug'] == 'noise' and pres and rng.random() < cfg['char_noise']:
        idx = int(rng.integers(0, len(parts)))
        parts[idx] = char_noise(parts[idx], rng)
    return parts


def build_train_texts(df_train, cfg, seed):
    rng = np.random.default_rng(seed)
    texts, labels = list(df_train['full_name']), list(df_train['label'])
    if cfg['aug'] != 'none':
        for _, row in df_train.iterrows():
            for _ in range(cfg['n_aug']):
                parts = augment_row(row['nom_clean'].split(),
                                    row['prenoms_clean'].split(), cfg, rng)
                if parts:
                    texts.append(' '.join(parts)); labels.append(row['label'])
    order = rng.permutation(len(texts))
    return [texts[i] for i in order], np.array(labels, dtype=np.float32)[order]

## 4. Encodage

Vocabulaires construits sur le **train uniquement** (le v4 incluait val et test).
- `flat` : chaîne complète de 36 caractères (v4).
- `by_word` : 8 mots × 14 caractères + identifiant de chaque mot (v5). Un mot vu moins de
  `word_min_freq` fois devient `<UNK>` : le modèle doit alors s'appuyer sur ses caractères.

In [ ]:
class Encoder:
    def __init__(self, train_texts, word_min_freq):
        chars = sorted(set(''.join(train_texts)) - {' '})
        self.char_to_index = {c: i+2 for i, c in enumerate(chars)}
        self.char_to_index.update({'<PAD>': 0, '<UNK>': 1, ' ': len(chars)+2})
        counts = Counter(w for t in train_texts for w in t.split())
        words = sorted(w for w, n in counts.items() if n >= word_min_freq)
        self.word_to_index = {w: i+2 for i, w in enumerate(words)}
        self.word_to_index.update({'<PAD>': 0, '<UNK>': 1})
        self.max_len = 36

    def flat(self, texts):
        X = np.zeros((len(texts), self.max_len), dtype=np.int32)
        for i, t in enumerate(texts):
            seq = [self.char_to_index.get(c, 1) for c in t][-self.max_len:]
            X[i, :len(seq)] = seq
        return X

    def by_word(self, texts):
        C = np.zeros((len(texts), MAX_TOKENS, MAX_CHARS), dtype=np.int32)
        W = np.zeros((len(texts), MAX_TOKENS), dtype=np.int32)
        for i, t in enumerate(texts):
            for j, w in enumerate(t.split()[:MAX_TOKENS]):
                W[i, j] = self.word_to_index.get(w, 1)
                seq = [self.char_to_index.get(c, 1) for c in w[:MAX_CHARS]]
                C[i, j, :len(seq)] = seq
        return C, W

## 5. Architectures

In [ ]:
def build_v4(cfg, vocab_size, max_len):
    """CNN + BiLSTM + MHA sur la chaîne complète (identique à train_model.ipynb)."""
    inp = L.Input(shape=(max_len,), dtype='int32', name='input')
    x = L.Embedding(vocab_size, cfg['embed_dim'])(inp)
    x = L.SpatialDropout1D(0.2)(x)
    c3 = L.Conv1D(cfg['cnn_filters'], 3, padding='same', activation='relu')(x)
    c5 = L.Conv1D(cfg['cnn_filters'], 5, padding='same', activation='relu')(x)
    x = L.Add()([c3, c5])
    x = L.Bidirectional(L.LSTM(cfg['lstm1'], return_sequences=True, dropout=0.2))(x)
    attn = L.MultiHeadAttention(cfg['mha_heads'], cfg['mha_key_dim'], dropout=0.1)(x, x)
    x = L.LayerNormalization()(L.Add()([x, attn]))
    x = L.Bidirectional(L.LSTM(cfg['lstm2'], dropout=0.2))(x)
    x = L.Dense(128, 'relu')(x); x = L.Dropout(0.4)(x)
    x = L.Dense(64, 'relu')(x);  x = L.Dropout(0.3)(x)
    out = L.Dense(1, 'sigmoid', name='output')(x)
    return tf.keras.Model(inp, out, name='CNN_BiLSTM_MHA_v4')


def build_v5(cfg, char_vocab, word_vocab):
    """Modèle par mot : encodeur de mot partagé + attention entre mots sans position."""
    chars_in = L.Input(shape=(MAX_TOKENS, MAX_CHARS), dtype='int32', name='chars')
    words_in = L.Input(shape=(MAX_TOKENS,), dtype='int32', name='words')
    tok_mask = L.Lambda(lambda w: tf.cast(w > 0, 'float32'), name='tok_mask')(words_in)

    # Forme du mot : n-grammes de caractères (k = 2, 3, 4) + max-pooling
    ce = L.Embedding(char_vocab, cfg['char_dim'])(chars_in)
    convs = [L.TimeDistributed(L.Conv1D(cfg['tok_dim'], k, padding='same',
                                        activation='relu'))(ce) for k in (2, 3, 4)]
    cv = L.Concatenate()([L.TimeDistributed(L.GlobalMaxPooling1D())(c) for c in convs])

    # Identité du mot (prénom fréquent), optionnelle : c'est la principale source de
    # mémorisation, d'où le dropout dédié et la possibilité de la retirer
    if cfg['use_words']:
        we = L.Embedding(word_vocab, cfg['word_dim'])(words_in)
        we = L.Dropout(cfg['word_dropout'])(we)
        h = L.Concatenate()([cv, we])
    else:
        h = cv
    h = L.Dense(cfg['tok_dim'], activation='relu')(h)
    h = L.Dropout(cfg['dropout'])(h)

    # Attention entre mots, sans encodage de position : invariance à l'ordre
    attn_mask = L.Lambda(lambda m: tf.einsum('bi,bj->bij', m, m) > 0)(tok_mask)
    for _ in range(cfg['n_attn_layers']):
        a = L.MultiHeadAttention(4, cfg['tok_dim'] // 4, dropout=0.1)(
            h, h, attention_mask=attn_mask)
        h = L.LayerNormalization()(L.Add()([h, a]))

    # Attention-pooling masqué : un poids par mot (interprétabilité)
    scores = L.Dense(1)(h)
    scores = L.Lambda(lambda z: z[0][..., 0] + (z[1] - 1.0) * 1e9)([scores, tok_mask])
    weights = L.Softmax(name='word_weights')(scores)
    pooled = L.Dot(axes=1)([weights, h])

    x = L.Dense(64, 'relu')(pooled)
    x = L.Dropout(cfg['dropout'])(x)
    out = L.Dense(1, 'sigmoid', name='output')(x)
    return tf.keras.Model([chars_in, words_in], out, name='WordAttn_v5')


_enc = Encoder(list(df_tr['full_name']), CFG['word_min_freq'])
print(f"Vocabulaire : {len(_enc.char_to_index)} caractères | {len(_enc.word_to_index):,} mots")
print(f"Paramètres v4 : {build_v4(CFG, len(_enc.char_to_index), 36).count_params():,}")
print(f"Paramètres v5 : {build_v5(CFG, len(_enc.char_to_index), len(_enc.word_to_index)).count_params():,}")

## 6. Entraînement

Recette commune aux deux architectures : AdamW, taux d'apprentissage en cosinus sur toute
la durée, arrêt anticipé sur `val_loss`, pas de lissage des étiquettes par défaut.

In [ ]:
def metrics(y, p):
    yp = (p > 0.5).astype(int)
    return dict(acc=float(accuracy_score(y, yp)),
                f1=float(f1_score(y, yp, average='macro')),
                auc=float(roc_auc_score(y, p)))


def optuna_prune_callback(trial):
    """Élagage Optuna : arrête un essai peu prometteur (sur val_auc)."""
    import optuna

    def on_epoch_end(epoch, logs):
        trial.report(logs['val_auc'], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return tf.keras.callbacks.LambdaCallback(on_epoch_end=on_epoch_end)


def run(cfg, seed=42, eval_test=False, trial=None, save_as=None, verbose=2,
        run_name=None, nested=False):
    """Entraîne une configuration et l'enregistre comme run MLflow.

    Paramètres, courbes par époque et métriques sont enregistrés. Un essai Optuna
    élagué est marqué KILLED (tag pruned=true) sans fermer le run parent.
    """
    mlrun = mlflow.start_run(run_name=run_name, nested=nested)
    mlflow.log_params({**cfg, 'seed': seed})
    mlflow.set_tags({'arch': cfg['arch'], 'aug': cfg['aug']})
    log_epoch = tf.keras.callbacks.LambdaCallback(on_epoch_end=lambda epoch, logs: (
        mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=epoch)))
    try:
        res = _train(cfg, seed, eval_test, trial, save_as, verbose, log_epoch)
    except BaseException as e:
        pruned = type(e).__name__ == 'TrialPruned'
        if pruned:
            mlflow.set_tag('pruned', 'true')
        mlflow.end_run(status='KILLED' if pruned else 'FAILED')
        raise
    mlflow.log_metrics({f'final_val_{k}': v for k, v in res['val'].items()})
    mlflow.log_metrics({k: res[k] for k in ('epochs_run', 'minutes', 'n_train', 'params')})
    if 'test' in res:
        mlflow.log_metrics({f'test_{k}': v for k, v in res['test'].items()})
    res['mlflow_run_id'] = mlrun.info.run_id
    mlflow.end_run()
    return res


def _train(cfg, seed, eval_test, trial, save_as, verbose, log_epoch):
    """Entraîne une configuration ; évalue la validation (et le test si eval_test)."""
    tf.keras.utils.set_random_seed(seed)
    tr_texts, y_tr = build_train_texts(df_tr, cfg, seed)
    enc = Encoder(list(df_tr['full_name']), cfg['word_min_freq'])
    y_va = df_va['label'].values.astype(np.float32)

    if cfg['arch'] == 'v4':
        model = build_v4(cfg, len(enc.char_to_index), enc.max_len)
        ds_tr = tf.data.Dataset.from_tensor_slices((enc.flat(tr_texts), y_tr))
        prep = enc.flat
    else:
        model = build_v5(cfg, len(enc.char_to_index), len(enc.word_to_index))
        C_tr, W_tr = enc.by_word(tr_texts)
        unk = cfg['unk_prob']

        def word_unk(x, y):
            c, w = x
            drop = tf.random.uniform(tf.shape(w)) < unk
            return (c, tf.where(drop & (w > 0), tf.ones_like(w), w)), y

        ds_tr = tf.data.Dataset.from_tensor_slices(((C_tr, W_tr), y_tr)).map(word_unk)
        prep = lambda texts: tuple(enc.by_word(texts))

    X_va = prep(list(df_va['full_name']))
    ds_tr = ds_tr.shuffle(20000, seed=seed).batch(cfg['batch_size']).prefetch(tf.data.AUTOTUNE)
    steps = int(np.ceil(len(y_tr) / cfg['batch_size'])) * cfg['epochs']
    lr = tf.keras.optimizers.schedules.CosineDecay(cfg['lr'], steps, alpha=0.01)
    model.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=cfg['weight_decay']),
                  loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=cfg['label_smooth']),
                  metrics=[tf.keras.metrics.AUC(name='auc'), 'accuracy'])

    callbacks = [tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=cfg['patience'], restore_best_weights=True), log_epoch]
    if trial is not None:
        callbacks.append(optuna_prune_callback(trial))

    t0 = time.time()
    hist = model.fit(ds_tr, validation_data=(X_va, y_va), epochs=cfg['epochs'],
                     callbacks=callbacks, verbose=verbose)
    p_va = model.predict(X_va, verbose=0).ravel()
    res = dict(cfg=cfg, seed=seed, n_train=len(y_tr), params=model.count_params(),
               epochs_run=len(hist.history['loss']), minutes=(time.time()-t0)/60,
               val=metrics(y_va, p_va), val_pred=p_va.tolist(),
               history={k: [float(v) for v in vs] for k, vs in hist.history.items()})
    if eval_test:
        p_te = model.predict(prep(list(df_te['full_name'])), verbose=0).ravel()
        res['test'] = metrics(df_te['label'].values, p_te)
        res['test_pred'] = p_te.tolist()
    if save_as:
        model.save(f'{save_as}.keras')
        with open(f'{save_as}_vocab.json', 'w') as f:
            json.dump(dict(char_to_index=enc.char_to_index, word_to_index=enc.word_to_index,
                           max_tokens=MAX_TOKENS, max_chars=MAX_CHARS, max_len=enc.max_len), f)
        if cfg['arch'] == 'v5':
            res['onnx_max_diff'] = export_onnx_v5(model, f'{save_as}.onnx', X_va, p_va)
    return res


def export_onnx_v5(model, path, X_va, p_keras):
    """Exporte le v5 en ONNX et renvoie l'écart maximal avec Keras sur la validation."""
    import tf2onnx, onnxruntime as ort
    spec = [tf.TensorSpec((None, MAX_TOKENS, MAX_CHARS), tf.int32, name='chars'),
            tf.TensorSpec((None, MAX_TOKENS), tf.int32, name='words')]
    tf2onnx.convert.from_keras(model, input_signature=spec, opset=15, output_path=path)
    sess = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    p_onnx = sess.run(None, {'chars': X_va[0], 'words': X_va[1]})[0].ravel()
    diff = float(np.abs(p_onnx - p_keras).max())
    print(f"ONNX exporté : {path} | écart max avec Keras = {diff:.2e}")
    return diff


def save(name, obj):
    with open(os.path.join(RUNS_DIR, f'{name}.json'), 'w') as f:
        json.dump(obj, f, indent=1)


def short(res):
    v = res['val']
    return (f"val acc={v['acc']:.4f} f1={v['f1']:.4f} auc={v['auc']:.4f} "
            f"({res.get('epochs_run', '-')} époques, {res.get('minutes', 0):.1f} min)")

## 7. Référence : régression logistique réglée

TF-IDF sur n-grammes de caractères + mots entiers, `C` et plage de n-grammes choisis sur la
validation. C'est le score à battre.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union


def lr_baseline(eval_test=False):
    """Grille sur la validation ; chaque configuration est un run MLflow enfant."""
    best = None
    with mlflow.start_run(run_name='lr_baseline_test' if eval_test else 'lr_baseline'):
        mlflow.set_tag('arch', 'LR')
        for ng in [(1, 5), (1, 6)]:
            for C in [1, 2, 3, 5]:
                pipe = make_pipeline(
                    make_union(TfidfVectorizer(analyzer='char_wb', ngram_range=ng, sublinear_tf=True),
                               TfidfVectorizer(analyzer='word', token_pattern=r'\S+')),
                    LogisticRegression(C=C, max_iter=3000))
                pipe.fit(df_tr['full_name'], df_tr['label'])
                m = metrics(df_va['label'].values, pipe.predict_proba(df_va['full_name'])[:, 1])
                with mlflow.start_run(run_name=f'lr_char{ng[0]}-{ng[1]}_C{C}', nested=True):
                    mlflow.log_params(dict(ngram=f'{ng[0]}-{ng[1]}', C=C))
                    mlflow.log_metrics({f'final_val_{k}': v for k, v in m.items()})
                print(f"  char{ng} C={C} : val acc={m['acc']:.4f} auc={m['auc']:.4f}")
                if best is None or m['acc'] > best[0]['acc']:
                    best = (m, ng, C, pipe)
        m, ng, C, pipe = best
        res = dict(model='LR', ngram=list(ng), C=C, val=m,
                   val_pred=pipe.predict_proba(df_va['full_name'])[:, 1].tolist())
        mlflow.log_params(dict(best_ngram=f'{ng[0]}-{ng[1]}', best_C=C))
        mlflow.log_metrics({f'final_val_{k}': v for k, v in m.items()})
        if eval_test:
            p = pipe.predict_proba(df_te['full_name'])[:, 1]
            res['test'] = metrics(df_te['label'].values, p)
            res['test_pred'] = p.tolist()
            mlflow.log_metrics({f'test_{k}': v for k, v in res['test'].items()})
    return res


if RUN_LR:
    mlflow.set_experiment('v5-reference')
    lr_res = lr_baseline()
    save('lr_baseline', lr_res)
    print(f"\nMeilleure LR : char{tuple(lr_res['ngram'])} C={lr_res['C']} -> {lr_res['val']}")

## 8. Entraînement unique (configuration `CFG`)

In [ ]:
if RUN_SINGLE:
    mlflow.set_experiment('v5-single')
    res = run(CFG, run_name=f"{CFG['arch']}_{CFG['aug']}")
    save(f"single_{CFG['arch']}_{CFG['aug']}", res)
    print(short(res))

## 9. Ablations : effet de l'augmentation sur v4 et v5

Même recette et même budget pour les 8 combinaisons. Permet de séparer l'effet de
l'augmentation de celui de l'architecture (tableau pour le mémoire).

In [ ]:
if RUN_ABLATION:
    mlflow.set_experiment('v5-ablation')
    ablation = {}
    with mlflow.start_run(run_name='ablation'):
        for arch in ('v4', 'v5'):
            for aug in ('v4', 'none', 'safe', 'noise'):
                r = run(dict(CFG, arch=arch, aug=aug), verbose=0,
                        run_name=f'{arch}/{aug}', nested=True)
                ablation[f'{arch}/{aug}'] = r
                print(f"{arch}/{aug:<6} {short(r)}")
                save('ablation', ablation)
    display(pd.DataFrame({k: r['val'] for k, r in ablation.items()}).T
              .sort_values('acc', ascending=False).round(4))

## 10. Recherche d'hyperparamètres (Optuna) sur le v5

Échantillonneur TPE, élagage des essais peu prometteurs (MedianPruner). L'étude est
enregistrée dans `runs/optuna_v5.db` : elle peut être interrompue puis reprise.

In [ ]:
if RUN_OPTUNA:
    import optuna

    def objective(trial):
        c = dict(CFG, arch='v5',
                 aug=trial.suggest_categorical('aug', ['none', 'safe', 'noise']),
                 n_aug=trial.suggest_int('n_aug', 1, 5),
                 word_drop=trial.suggest_float('word_drop', 0.0, 0.5),
                 char_dim=trial.suggest_categorical('char_dim', [32, 48, 64]),
                 word_dim=trial.suggest_categorical('word_dim', [32, 64, 128]),
                 tok_dim=trial.suggest_categorical('tok_dim', [64, 128, 192]),
                 n_attn_layers=trial.suggest_int('n_attn_layers', 0, 2),
                 word_min_freq=trial.suggest_int('word_min_freq', 1, 4),
                 unk_prob=trial.suggest_float('unk_prob', 0.0, 0.5),
                 dropout=trial.suggest_float('dropout', 0.1, 0.5),
                 lr=trial.suggest_float('lr', 3e-4, 3e-3, log=True),
                 batch_size=trial.suggest_categorical('batch_size', [64, 128, 256]),
                 label_smooth=trial.suggest_categorical('label_smooth', [0.0, 0.05]))
        r = run(c, trial=trial, verbose=0, run_name=f'essai_{trial.number}', nested=True)
        print(f"essai {trial.number}: {short(r)}")
        return r['val']['acc']

    mlflow.set_experiment('v5-optuna')
    study = optuna.create_study(
        direction='maximize', study_name='v5',
        storage=f'sqlite:///{RUNS_DIR}/optuna_v5.db', load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
    with mlflow.start_run(run_name=f'optuna_{N_TRIALS}_essais'):
        study.optimize(objective, n_trials=N_TRIALS,
                       timeout=OPTUNA_TIMEOUT_H * 3600 if OPTUNA_TIMEOUT_H else None)
        mlflow.log_params({f'best_{k}': v for k, v in study.best_params.items()})
        mlflow.log_metric('best_val_acc', study.best_value)
    best_cfg = dict(CFG, arch='v5', **study.best_params)
    save('best_v5', best_cfg)
    print(f"Meilleure val acc : {study.best_value:.4f}")
    display(study.trials_dataframe().sort_values('value', ascending=False).head(10))

## 11. Évaluation finale sur le test (une seule fois)

Configuration figée (`runs/best_v5.json` si elle existe, sinon `CFG`), 3 graines,
comparaison avec la régression logistique (test de McNemar) et ensemble des deux.

In [ ]:
if RUN_FINAL:
    from scipy.stats import binomtest
    path = os.path.join(RUNS_DIR, 'best_v5.json')
    final_cfg = json.load(open(path)) if FINAL_USE_BEST and os.path.exists(path) else dict(CFG)
    os.makedirs(FINAL_DIR, exist_ok=True)
    y_te = df_te['label'].values

    mlflow.set_experiment('v5-final')
    lr_final = lr_baseline(eval_test=True)
    p_lr = np.array(lr_final.pop('test_pred'))
    p_lr_va = np.array(lr_final.pop('val_pred'))
    y_va = df_va['label'].values
    final = dict(cfg=final_cfg, lr=lr_final, runs=[])
    for s in FINAL_SEEDS:
        r = run(final_cfg, seed=s, eval_test=True, verbose=0, run_name=f'final_seed{s}',
                save_as=f"{RUNS_DIR}/model_{final_cfg['arch']}_seed{s}")
        p = np.array(r.pop('test_pred'))
        np.savez(os.path.join(FINAL_DIR, f"{final_cfg['arch']}_seed{s}.npz"),
                 val=np.array(r['val_pred']), test=p)
        e_m, e_l = (p > 0.5) != y_te, (p_lr > 0.5) != y_te
        b, c = int((e_m & ~e_l).sum()), int((~e_m & e_l).sum())
        r['mcnemar_vs_lr'] = dict(modele_faux_lr_juste=b, modele_juste_lr_faux=c,
                                  p=float(binomtest(min(b, c), b + c, 0.5).pvalue))
        r['ensemble_avec_lr'] = metrics(y_te, (p + p_lr) / 2)
        r['ensemble_avec_lr_val'] = metrics(y_va, (np.array(r.pop('val_pred')) + p_lr_va) / 2)
        with mlflow.start_run(run_id=r['mlflow_run_id']):
            mlflow.log_metrics({'mcnemar_p': r['mcnemar_vs_lr']['p'],
                                'mcnemar_modele_faux_lr_juste': b,
                                'mcnemar_modele_juste_lr_faux': c,
                                **{f'ensemble_test_{k}': v for k, v in r['ensemble_avec_lr'].items()},
                                **{f'ensemble_val_{k}': v for k, v in r['ensemble_avec_lr_val'].items()}})
        final['runs'].append(r)
        print(f"graine {s} : test {r['test']} | McNemar p={r['mcnemar_vs_lr']['p']:.4f}")
        save(f"final_{final_cfg['arch']}", final)

    accs = [r['test']['acc'] for r in final['runs']]
    print(f"\n{final_cfg['arch']} : test acc = {np.mean(accs):.4f} ± {np.std(accs):.4f}"
          f" | LR : {lr_final['test']['acc']:.4f}")

## 12. Synthèse des résultats enregistrés

In [ ]:
rows = []
for fname in sorted(os.listdir(RUNS_DIR)):
    if not fname.endswith('.json') or fname == 'best_v5.json': continue
    obj = json.load(open(os.path.join(RUNS_DIR, fname)))
    items = obj.items() if fname == 'ablation.json' else [(fname[:-5], obj)]
    for name, r in items:
        if isinstance(r, dict) and 'val' in r:
            rows.append(dict(experience=name, **{f'val_{k}': v for k, v in r['val'].items()}))
if rows:
    display(pd.DataFrame(rows).sort_values('val_acc', ascending=False).round(4))
else:
    print('Aucun résultat enregistré dans', RUNS_DIR)